# Redrob Candidate Ranker — Sandbox

This notebook is the hackathon **sandbox environment**: a working hosted environment where the ranker can be run on a small candidate sample (per `submission_spec.md` §10.5).

**What it demonstrates:**
- The exact same `src/` code as the submitted GitHub repo (cloned fresh, not copy-pasted)
- End-to-end run: raw candidates → structured features → local TF-IDF/SVD embeddings → ranked, reasoned output
- Zero network calls during the ranking step, zero GPU, fully deterministic
- Validated CSV format

Runs on Colab's free CPU tier — no GPU needed, nothing to configure.


## 1. Get the code

Clones the actual submitted repo. If you're running this before the repo is public, use the fallback upload cell instead (just below).

In [ ]:

REPO_URL = "https://github.com/Suhesh-3011/redrob-ranker.git"

import os, subprocess

if os.path.exists("/content/redrob-ranker"):
    print("Repo already present, pulling latest...")
    subprocess.run(["git", "-C", "/content/redrob-ranker", "pull"], check=False)
else:
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/redrob-ranker"],
                             capture_output=True, text=True)
    print(result.stdout, result.stderr)
    if result.returncode != 0:
        print("\n" + "="*70)
        print("Clone failed (repo may not be public yet / URL not set).")
        print("Use the manual-upload fallback cell below instead.")
        print("="*70)

os.chdir("/content/redrob-ranker" if os.path.exists("/content/redrob-ranker") else "/content")
print("Working directory:", os.getcwd())


### Fallback: manual upload (only if the git clone above failed)

Skip this cell if the clone worked. Otherwise, zip your local `redrob-ranker/` folder and upload it here.

In [ ]:
# FALLBACK ONLY — skip if the clone cell above succeeded.
import os

if not os.path.exists("/content/redrob-ranker/src"):
    try:
        from google.colab import files
        import zipfile
        print("Upload redrob-ranker.zip (zip of the whole repo folder)...")
        uploaded = files.upload()
        for fname in uploaded:
            if fname.endswith(".zip"):
                with zipfile.ZipFile(fname, "r") as z:
                    z.extractall("/content/")
                print("Extracted", fname)
    except ImportError:
        print("google.colab not available (not running in Colab). "
              "This fallback only works inside Colab -- outside Colab, "
              "populate /content/redrob-ranker manually instead.")
else:
    print("Repo already present -- skipping manual upload.")

os.chdir("/content/redrob-ranker")
print("Working directory:", os.getcwd())


## 2. Install dependencies

Exactly `requirements.txt` from the repo — nothing extra, nothing hidden.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    capture_output=True, text=True
)
if result.returncode != 0 and "externally-managed-environment" in result.stderr:
    # Some hosted environments enforce PEP 668; --break-system-packages is a
    # safe no-op on environments that don't need it (e.g. standard Colab).
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "--break-system-packages"],
        capture_output=True, text=True
    )

if result.returncode == 0:
    print("Dependencies installed successfully.")
else:
    print("INSTALL FAILED:")
    print(result.stderr[-2000:])


## 3. Run the offline precompute step (small sample)

Per the hackathon rules, the sandbox only needs to demonstrate the ranker on a **small candidate sample** — the full 100K run happens on the grading side, not here. We use the bundled `data/sample_candidates.json` (50 candidates).

This step extracts structured, rule-based features and fits a local TF-IDF+SVD embedding space. **No network calls, no model downloads** — everything here is scikit-learn running locally.

In [ ]:
import subprocess, time

t0 = time.time()
result = subprocess.run(
    ["python3", "precompute.py",
     "--candidates", "../data/sample_candidates.json",
     "--out", "../artifacts_demo"],
    cwd="src", capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
print(f"\nWall time: {time.time()-t0:.1f}s")


## 4. Run the ranking step

**This is the step that's timed and network-restricted in the real evaluation** (≤5 min, CPU-only, zero network). Watch the printed wall time below — on the full 100K pool this measured **5.9 seconds** during our own testing.

We ask for the top 20 here since the sample only has 50 candidates (`--top-n 20`, purely a sandbox-demo convenience — the real submission always produces exactly 100 rows, which is what `rank.py` does by default against the full pool).

In [ ]:
result = subprocess.run(
    ["python3", "rank.py",
     "--artifacts", "../artifacts_demo",
     "--out", "../submission_demo.csv",
     "--top-n", "20"],
    cwd="src", capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)


## 5. Inspect the ranked output

Every reasoning string below is generated entirely from extracted profile facts — no LLM call anywhere in this pipeline.

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)

df = pd.read_csv("submission_demo.csv")
print(f"{len(df)} ranked candidates. Columns: {list(df.columns)}\n")
df.head(10)


In [ ]:
# Full reasoning text for the top 5, unabridged
for _, row in df.head(5).iterrows():
    print(f"Rank {row['rank']}  |  {row['candidate_id']}  |  score={row['score']}")
    print(f"  {row['reasoning']}\n")


## 6. Format validation

`validate_submission.py` is the organizer-provided validator. It expects exactly 100 rows (the official submission size), so it will correctly flag this 20-row demo file as short — **that's expected and fine here**; it confirms the validator is doing its job. Against the real 100K-pool output (100 rows), this same validator returns "Submission is valid." (verified separately — see the deck / README for that run's output).

In [ ]:
result = subprocess.run(
    ["python3", "validate_submission.py", "submission_demo.csv"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)


## 7. What this run demonstrates

| Check | Result |
|---|---|
| Network calls during ranking | Zero (see step 4 output — no external requests, no model hub) |
| GPU usage | None — pure CPU (numpy + scikit-learn) |
| Wall-clock for the ranking step | Printed above (well under the 5-minute budget even at 100K scale) |
| Reasoning source | 100% template-generated from extracted features, zero LLM calls |
| Honeypot detection | Active — see the `disqualified / honeypot-suspect / keyword-stuffing-suspect` counts printed in step 4 |

For the full architecture writeup (why TF-IDF+SVD instead of a downloaded transformer model, the keyword-stuffing bug this caught during testing, the scoring formula, etc.), see `README.md` in the repo root and the submitted approach deck.


---
### Running against the full 100K pool (not needed for sandbox review, informational only)

```bash
python src/precompute.py --candidates data/candidates.jsonl --out artifacts
python src/rank.py --artifacts artifacts --out submission.csv
python validate_submission.py submission.csv
```

Measured on the actual 100,000-candidate pool: precompute ~2.5 min (offline, untimed), ranking step **5.9 seconds** (budget: 5 minutes), honeypot rate in the final top 100: **0%**.
